In [1]:
import numpy as np
from collections import deque

In [ ]:
# 1. BASIC FUNCTIONS

def euclidean_distance(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))


def route_cost(route, points):
    total = 0.0

    for i in range(len(route)):
        a = points[route[i]]
        b = points[route[(i + 1) % len(route)]]

        total += euclidean_distance(a, b)

    return total

In [ ]:
# 2. CONSTRUCTIVE HEURISTICS

def nearest_neighbor(points, start=0):
    n = len(points)

    if n <= 1:
        return list(range(n))

    route = [start]

    unvisited = set(range(n))
    unvisited.remove(start)

    current = start

    while unvisited:

        next_node = min(
            unvisited,
            key=lambda node:
                euclidean_distance(
                    points[current],
                    points[node]
                )
        )

        route.append(next_node)
        unvisited.remove(next_node)

        current = next_node

    return route


def nearest_insertion(points, start=0):
    n = len(points)

    if n <= 1:
        return list(range(n))

    if n == 2:
        return [start, 1 - start]

    remaining = set(range(n))
    remaining.remove(start)

    # Cari node terdekat dari start
    nearest = min(
        remaining,
        key=lambda node:
            euclidean_distance(
                points[start],
                points[node]
            )
    )

    route = [start, nearest]
    remaining.remove(nearest)

    while remaining:

        # Cari node yang paling dekat dengan tour
        node = min(
            remaining,
            key=lambda candidate:
                min(
                    euclidean_distance(
                        points[candidate],
                        points[r]
                    )
                    for r in route
                )
        )

        best_position = None
        best_increase = float("inf")

        # Cari posisi insertion terbaik
        for i in range(len(route)):

            a = route[i]
            b = route[(i + 1) % len(route)]

            increase = (
                euclidean_distance(
                    points[a],
                    points[node]
                )
                + euclidean_distance(
                    points[node],
                    points[b]
                )
                - euclidean_distance(
                    points[a],
                    points[b]
                )
            )

            if increase < best_increase:
                best_increase = increase
                best_position = i + 1

        route.insert(best_position, node)
        remaining.remove(node)

    return route


def farthest_neighbor(points, start=0):
    n = len(points)

    if n <= 1:
        return list(range(n))

    route = [start]

    unvisited = set(range(n))
    unvisited.remove(start)

    current = start

    while unvisited:

        next_node = max(
            unvisited,
            key=lambda node:
                euclidean_distance(
                    points[current],
                    points[node]
                )
        )

        route.append(next_node)
        unvisited.remove(next_node)

        current = next_node

    return route


def arbitrary_solution(points, seed=None):
    """
    Arbitrary solution:
    Urutan node diacak.
    """
    route = np.arange(len(points))

    rng = np.random.default_rng(seed)
    rng.shuffle(route)

    return route.tolist()

In [ ]:
# 3. 2-OPT NEIGHBORHOOD

def generate_2opt_neighbors(route):
    neighbors = []
    n = len(route)

    if n < 4:
        return neighbors

    for i in range(1, n - 1):

        for k in range(i + 1, n):

            neighbor = (
                route[:i]
                + route[i:k + 1][::-1]
                + route[k + 1:]
            )

            move = (i, k)

            neighbors.append(
                (neighbor, move)
            )

    return neighbors


def random_2opt_neighbor(route, rng):
    """
    Mengambil satu neighbor 2-opt secara random.
    """
    n = len(route)

    if n < 4:
        return route[:], (None, None)

    # Ambil dua index berbeda
    i, k = np.sort(
        rng.choice(
            np.arange(1, n),
            size=2,
            replace=False
        )
    )

    neighbor = (
        route[:i]
        + route[i:k + 1][::-1]
        + route[k + 1:]
    )

    return neighbor, (int(i), int(k))

In [ ]:
# 4. TABU SEARCH

def tabu_search(
    points,
    initial_route,
    max_iterations=100,
    tabu_tenure=10
):

    current = initial_route[:]
    current_cost = route_cost(current, points)

    best = current[:]
    best_cost = current_cost

    tabu_list = deque(maxlen=tabu_tenure)

    history = []

    for iteration in range(max_iterations):

        neighbors = generate_2opt_neighbors(current)

        if not neighbors:
            break

        selected_route = None
        selected_cost = float("inf")
        selected_move = None

        for candidate, move in neighbors:

            candidate_cost = route_cost(
                candidate,
                points
            )

            is_tabu = move in tabu_list

            # Aspiration:
            # boleh menerima move tabu kalau
            # menghasilkan best solution baru.
            aspiration = candidate_cost < best_cost

            if is_tabu and not aspiration:
                continue

            if candidate_cost < selected_cost:

                selected_route = candidate
                selected_cost = candidate_cost
                selected_move = move

        if selected_route is None:
            break

        current = selected_route
        current_cost = selected_cost

        tabu_list.append(selected_move)

        if current_cost < best_cost:

            best = current[:]
            best_cost = current_cost

        history.append({
            "iteration": iteration + 1,
            "current_cost": current_cost,
            "best_cost": best_cost
        })

    return best, best_cost, history

In [ ]:
# 5. SIMULATED ANNEALING

def simulated_annealing(
    points,
    initial_route,
    initial_temperature=1000.0,
    cooling_rate=0.995,
    min_temperature=0.001,
    iterations_per_temperature=100,
    seed=None
):

    rng = np.random.default_rng(seed)

    current = initial_route[:]
    current_cost = route_cost(current, points)

    best = current[:]
    best_cost = current_cost

    temperature = initial_temperature

    history = []

    while temperature > min_temperature:

        for _ in range(iterations_per_temperature):

            candidate, _ = random_2opt_neighbor(
                current,
                rng
            )

            candidate_cost = route_cost(
                candidate,
                points
            )

            delta = candidate_cost - current_cost

            # Solution lebih baik
            if delta < 0:

                accept = True

            else:

                # Probabilitas menerima solusi lebih buruk
                probability = np.exp(
                    -delta / temperature
                )

                accept = rng.random() < probability

            if accept:

                current = candidate
                current_cost = candidate_cost

                if current_cost < best_cost:

                    best = current[:]
                    best_cost = current_cost

        history.append({
            "temperature": temperature,
            "best_cost": best_cost
        })

        # Cooling
        temperature *= cooling_rate

    return best, best_cost, history

In [ ]:
# 6. MAIN SOLVER

def solve_tsp(
    points,
    initial_method="nearest_neighbor",
    improvement_method=None,
    start=0,
    **kwargs
):
    """
    Flow utama:

    points
       ↓
    Constructive heuristic
       ↓
    Initial solution
       ↓
    Improvement (optional)
       ↓
    Final solution
    """

    # STEP 1
    # Initial Solution

    if initial_method == "nearest_neighbor":

        initial_route = nearest_neighbor(
            points,
            start
        )

    elif initial_method == "nearest_insertion":

        initial_route = nearest_insertion(
            points,
            start
        )

    elif initial_method == "farthest_neighbor":

        initial_route = farthest_neighbor(
            points,
            start
        )

    elif initial_method == "arbitrary":

        initial_route = arbitrary_solution(
            points,
            seed=kwargs.get("seed")
        )

    else:

        raise ValueError(
            f"Unknown initial_method: {initial_method}"
        )

    initial_cost = route_cost(
        initial_route,
        points
    )

    # STEP 2
    # Improvement

    if improvement_method is None:

        final_route = initial_route
        final_cost = initial_cost
        history = []

    elif improvement_method == "tabu_search":

        final_route, final_cost, history = tabu_search(
            points,
            initial_route,
            max_iterations=kwargs.get(
                "max_iterations",
                100
            ),
            tabu_tenure=kwargs.get(
                "tabu_tenure",
                10
            )
        )

    elif improvement_method == "simulated_annealing":

        final_route, final_cost, history = simulated_annealing(
            points,
            initial_route,
            initial_temperature=kwargs.get(
                "initial_temperature",
                1000.0
            ),
            cooling_rate=kwargs.get(
                "cooling_rate",
                0.995
            ),
            min_temperature=kwargs.get(
                "min_temperature",
                0.001
            ),
            iterations_per_temperature=kwargs.get(
                "iterations_per_temperature",
                100
            ),
            seed=kwargs.get("seed")
        )

    else:

        raise ValueError(
            f"Unknown improvement_method: {improvement_method}"
        )

    # STEP 3
    # Return

    return {
        "initial_route": initial_route,
        "initial_cost": initial_cost,
        "final_route": final_route,
        "final_cost": final_cost,
        "history": history
    }

In [ ]:
# Contoh Penggunaan Dengan List
points = [
    [0, 0],   # index 0
    [2, 3],   # index 1
    [5, 1],   # index 2
    [6, 5],   # index 3
    [8, 2]    # index 4
]

result = solve_tsp(
    points,
    initial_method="nearest_neighbor",
    improvement_method="tabu_search",
    max_iterations=100,
    tabu_tenure=10
)

print("Initial route :", result["initial_route"])
print("Initial cost  :", result["initial_cost"])

print("Final route   :", result["final_route"])
print("Final cost    :", result["final_cost"])

Initial route : [0, 1, 2, 4, 3]
Initial cost  : 21.789181162467003
Final route   : [0, 2, 4, 3, 1]
Final cost    : 19.94453567968872


In [ ]:
# Contoh Penggunaan Dengan Numpy Array
points = np.array([
    [0, 0],
    [2, 3],
    [5, 1],
    [6, 5],
    [8, 2]
])

result = solve_tsp(
    points,
    initial_method="nearest_insertion",
    improvement_method="simulated_annealing",
    initial_temperature=1000,
    cooling_rate=0.995,
    min_temperature=0.001,
    iterations_per_temperature=100,
    seed=42
)

print("Initial route :", result["initial_route"])
print("Initial cost  :", result["initial_cost"])

print("Final route   :", result["final_route"])
print("Final cost    :", result["final_cost"])

Initial route : [0, 2, 4, 3, 1]
Initial cost  : 19.94453567968872
Final route   : [0, 2, 4, 3, 1]
Final cost    : 19.94453567968872


In [ ]:
# Contoh Penggunaan Dengan Nama Titik
points = np.array([
    [0, 0],
    [2, 3],
    [5, 1],
    [6, 5],
    [8, 2]
])

names = [
    "Jakarta",
    "Bandung",
    "Yogyakarta",
    "Surabaya",
    "Malang"
]


result = solve_tsp(
    points,

    # Initial solution
    initial_method="nearest_neighbor",

    # Improvement
    improvement_method="tabu_search",

    # Parameter Tabu Search
    max_iterations=100,
    tabu_tenure=10
)


print("Initial route:")
print(result["initial_route"])

print("Initial cost:")
print(result["initial_cost"])

print()

print("Final route:")
print(result["final_route"])

print("Final cost:")
print(result["final_cost"])

named_route = [
    names[i]
    for i in result["final_route"]
]

# Kembali ke titik awal
named_route.append(named_route[0])

print()
print("Final route (name):")

for city in named_route:
    print(city)

Initial route:
[0, 1, 2, 4, 3]
Initial cost:
21.789181162467003

Final route:
[0, 2, 4, 3, 1]
Final cost:
19.94453567968872

Final route (name):
Jakarta
Yogyakarta
Malang
Surabaya
Bandung
Jakarta
